# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Detección de Argumentos con ollama_chat/gemma3:27b en poliGPT API

In [1]:
%pip install pydantic pandas langchain numpy pymupdf openai openpyxl --quiet

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [41]:
from typing import List
from pydantic import BaseModel, Field, ValidationError
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.exceptions import OutputParserException
import requests
import json
import re
from openai import OpenAI
import openai
import httpx
import pandas as pd
import numpy as np
import os
import openpyxl

process_text_path = "..\\Data\\Processed Files (sections)\\"

model_name="gemma3:27b"
prefix = 'GLOBAL_SGD2025_'
output_dir = "..\\Data\\Extracted Arguments Keywords (all text)\\"

## Input text processing

In [42]:
# 1. Define your Pydantic schema for output
class ArgumentResponse(BaseModel):
    arguments: List[str] = Field(..., description="List of arguments extracted directly from the text.")

# 2. Setup output parser
pydantic_parser = PydanticOutputParser(pydantic_object=ArgumentResponse)

# 3. Extend text with first sentence from the next page
def extend_pages_with_next_sentence(pages):
    def get_first_sentence(text):
        match = re.search(r'(.+?\.)', text.strip())
        return match.group(1).strip() if match else ""

    extended_pages = []
    for i, page in enumerate(pages):
        current_text = page["text"]
        if i + 1 < len(pages):
            next_sentence = get_first_sentence(pages[i + 1]["text"])
            current_text += " " + next_sentence
        extended_pages.append({
            "page": page["page"],
            "text": current_text
        })
    return extended_pages

# 4. Build the prompt and call the LLM to extract arguments
def extract_arguments_json(text, topic, keywords, model_name) -> ArgumentResponse:
    format_instructions = pydantic_parser.get_format_instructions()

    # Keywords for filtering arguments
    positive_keywords = (keywords or {}).get('in_favor', [])
    negative_keywords = (keywords or {}).get('against', [])

    # Normalize & join for readability
    def to_str(xs):
        return ", ".join(sorted({s.strip().lower() for s in xs if isinstance(s, str) and s.strip()}))
    pos_kw_str = to_str(positive_keywords)
    neg_kw_str = to_str(negative_keywords)

    prompt = PromptTemplate(
        template=(
            "Task: Text Span Identification for Arguments related ONLY to Sustainable Development Goal: {topic}\n"

            "Role: You are an expert in logical reasoning, sustainability reporting, and argument analysis. \n"
            "Your job is to identify and extract verbatim arguments about {topic} from long-form sustainability texts.\n\n"

            "Instructions:\n"
            "1. Carefully read the entire input text.\n"
            "2. Identify ONLY those sentences or phrases that:\n"
            "   - Clearly support or argue for or against the topic {topic}\n"
            "   - Contain keyword from the relevant lists below\n"
            "   - Are exclusively about {topic} (EXCLUDE if they mention or refer to other SDGs or unrelated sustainability topics)\n\n"
            "3. Keywords for filtering:\n"
            "   - In favor: {pos_kw_str}\n"
            "   - Against: {neg_kw_str}\n"
            "4. Each extracted argument must:\n"
            "   - Relate exclusively to the specified SDG ({topic})\n"
            "   - Stand as a full statement\n"
            "   - Be copied exactly from the original (no paraphrasing)\n"
            "   - Include only the necessary context for understanding\n"
            "5. If no qualifying arguments are found, return an empty array.\n\n"

            "Output Rules:\n"
            "   - Use only the exact text from the original\n"
            "   - No additional commentary or explanation\n"
            "   - Return only valid JSON\n"
            "   - No markdown formatting\n\n"

            "Text:\n\"\"\"\n{text}\n\"\"\"\n\n"

            "Respond ONLY with a JSON object like this:\n\n"
            "{format_instructions}"
        ),
        input_variables=["text", "topic"],
        partial_variables={
            "format_instructions": format_instructions,
            "pos_kw_str": pos_kw_str,
            "neg_kw_str": neg_kw_str,
        },
    )

    final_prompt = prompt.format_prompt(text=text, topic=topic).to_string()

    client = OpenAI(
    base_url = 'https://api.poligpt.upv.es',  
    api_key = 'sk-Icbf-5FyeV0QcLWBC9SNEA',
    timeout=180     
        )

    timeout = httpx.Timeout(60.0, connect=30.0) 

    chat_completion = client.chat.completions.create(
        messages = [
            {'role': 'system', 'content': 'You are an expert in logical reasoning, sustainability reporting, and argument analysis.'},
            {'role': 'user', 'content': final_prompt}
        ],
        model = model_name,
        temperature = 0,
        # timeout = timeout
    )

    raw_output = chat_completion.choices[0].message.content

    try:
        return pydantic_parser.parse(raw_output)
    except OutputParserException as err:
        print("Parse failed:", err)
        return ArgumentResponse(arguments=[])

# 5. Wrapper function for pipeline
def extract_arguments_from_text(text, topic, keywords, model_name) -> List[str]:
    result = extract_arguments_json(text, topic, keywords, model_name)
    return result.arguments

# 6. Main document-level processor
def process_document(pages, model_name, topic="", keywords=None):
    extended_pages = extend_pages_with_next_sentence(pages)
    processed = []
    for page in extended_pages:
        print(f"\n--- Processing Page {page['page']} ---")
        #print("Text to analyze:\n", page["text"])
        
        arguments = extract_arguments_from_text(page["text"], topic, keywords, model_name)
        
        print("Extracted Arguments:")
        for i, arg in enumerate(arguments, 1):
            print(f"{i}. {arg}")

        processed.append({
            "page": page["page"],
            "text": page["text"],
            "arguments": arguments
        })
    return processed


# 7. File I/O
def save_to_json(processed, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(processed, f, indent=2, ensure_ascii=False)

def process_directory(input_dir, output_dir, prefix, model_name, topic="", keywords=None):
    os.makedirs(output_dir, exist_ok=True)
    all_results = []

    for filename in os.listdir(input_dir):
        if filename.endswith(".json") and filename.startswith(prefix):
            filepath = os.path.join(input_dir, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                pages = json.load(f)

            section_name = filename.replace(".json", "")
            processed = process_document(pages, model_name, topic, keywords)

            for item in processed:
                item["section"] = section_name  # Add section identifier
                all_results.append(item)
                
    return all_results

## SGD 1: Poverty

In [43]:
topic = "SGD 1 (Poverty): End poverty in all its forms everywhere"
sgd_number = "1"

keywords_g1 = {
    "in_favor": [
        "poverty reduction", "poverty alleviation", "social protection", "economic empowerment",
        "wealth creation", "opportunity", "prosperity", "development aid", "microfinance",
        "basic income", "empowerment", "upliftment", "sufficiency", "inclusion", "equity"
    ],
    "against": [
        "poverty", "pennilessness", "distress", "necessity", "hardship", "insolvency",
        "privation", "penury", "destitution", "hand-to-mouth existence", "beggary",
        "indigence", "pauperism", "necessitousness", "extreme poverty", "wealth inequality",
        "exploitation", "lack of opportunity", "exclusion", "vulnerability",
        "deprivation", "marginalization"
    ]
}


resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g1)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:
1. For many developing countries, a lack of fiscal space is the major obstacle to SDG progress.
2. Roughly half the world’s population lives in countries that cannot invest adequately in sustainable development due to debt burdens and a lack of access to affordable, long-term capital.

--- Processing Page 13 ---
Extracted Arguments:
1. Since 2016, SDG financing from official sources has received remarkably short shrift.
2. the high-income countries have delayed critical capital increases at the World Bank and other multilateral development banks, even though the SDG financing gap is large and well documented
3. the high-income countries have delayed critical increases in International Monetary Fund quotas and Special Drawing Rights allocations.
4. the upcoming Fourth International Conference on Financing for Development (FfD4), in Seville, Spain from June 30 – July 3, 2025, should send a m

## SGD 2: Hunger

In [44]:
topic = "SGD 2 (Hunger): End hunger, achieve food security and improved nutrition and promote sustainable agriculture"
sgd_number = "2"
keywords_g2 = {
    "in_favor": [
        "food security", "food", "nutrition", "zero hunger", "nourishment",
        "food sovereignty", "food aid", "school feeding programs",
        "access to food", "healthy diets"
    ],
    "against": [
        "hunger", "undernutrition", "malnutrition", "starvation", "famine",
        "undernourishment", "food insecurity", "food waste", "crop failure",
        "land grabbing", "price volatility", "nutrient deficiency"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g2)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:
1. an estimated 733 million people struggle with chronic hunger
2. roughly a third of humanity cannot afford a healthy diet

--- Processing Page 16 ---
Extracted Arguments:
1. “If we really wish to prepare a path to peace in our world, let us commit ourselves to remedying the remote causes of injustice, settling unjust and unpayable debts, and feeding the hungry.”

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:
1. Smallholder farmers in rural areas constitute roughly three-quarters of those living in extreme income poverty and over 83 percent of multidimensionally poor people.
2. They can best be supported in their livelihoods and wellbeing by prog

## SGD 3: Health

In [45]:
topic = "SGD 3 (Health): Ensure healthy lives and promote well-being for all at all ages"
sgd_number = "3"
keywords_g3 = {
    "in_favor": [
        "wellbeing", "welfare", "health", "benefit", "advantage", "comfort",
        "happiness", "prosperity", "universal health coverage", "healthcare access",
        "disease prevention", "mental health", "healthy lifestyles", "vaccination",
        "maternal health", "child health", "sanitation", "public health", "interest"
    ],
    "against": [
        "disease", "illness", "epidemic", "pandemic", "mortality", "morbidity",
        "health inequality", "stress", "poor sanitation", "addiction",
        "unhealthy habits", "mental illness", "anxiety"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g3)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Extracted Arguments:
1. under-5 mortality rate (SDG 3)
2. neonatal mortality (SDG 3)

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:
1. The UN budget must be met in full, and indeed increased.
2. Efficiencies in UN operations are to be welcomed, but cutting UN budgets at a time of pervasive conflicts, human displacements, climate disasters, epidemic diseases, and other crises is unacceptable.

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:
1. In addition to investing in the planet’s environmental sustainability, the most reliably high return on the planet comes from investing in the health and education of a young child in a low-income country in Africa, Asia, Oceania, or Latin America and the Caribbean.
2. Education not only fosters dignity, fulfillment, and wellbeing, but also delivers remarkable and reliable economic benefits; leading economists to describe

## SGD 4: Education

In [46]:
topic = "SGD 4 (Education): Ensure inclusive and equitable quality education"
sgd_number = "4"
keywords_g4 = {
    "in_favor": [
        "quality", "inclusive", "equitable", "lifelong learning", "teaching",
        "schooling", "training", "development", "coaching", "instruction",
        "tutoring", "tuition", "skills development", "literacy", "numeracy",
        "universal access", "scholarships", 'data literacy'
    ],
    "against": [
        "lack of education", "illiteracy", "school dropout", "dropout",
        "educational inequality", "poor quality teaching", "indoctrination",
        "lack of access", "resource scarcity", "digital divide", "skills gap"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g4)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:
1. In addition to investing in the planet’s environmental sustainability, the most reliably high return on the planet comes from investing in the health and education of a young child in a low-income country in Africa, Asia, Oceania, or Latin America and the Caribbean.
2. Education not only fosters dignity, fulfillment, and wellbeing, but also delivers remarkable and reliable economic benefits; leading economists to describe healthcare, nutrition, and education as investments in human capital.
3. Such investments have a huge financial payoff with perhaps a 20 percent compound annual return when they are broad-based and of good quality.
4. The most pressing practical challenge is to enable such investments even in impoverished areas where governm

## SGD 5: Gender

In [ ]:
topic = "SGD 5 (Gender): Achieve gender equality and empower all women and girls"
sgd_number = "5"
keywords_g5 = {
    "in_favor": [
        "gender equality", "women empowerment", "feminism", "women’s movement",
        "suffragette", "suffragist", "feminist", "emancipated", "equal rights",
        "equal opportunity", "women leadership", "girls education", "reproductive rights"
    ],
    "against": [
        "gender inequality", "sexism", "sexist", "discrimination", "gender violence",
        "misogyny", "patriarchy", "wage gap", "glass ceiling", "female genital mutilation",
        "child marriage", "lack of representation", "stereotypes", "glass ceiling"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g5)


merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:
1. Education not only fosters dignity, fulfillment, and wellbeing, but also delivers remarkable and reliable economic benefits; leading economists to describe healthcare, nutrition, and education as investments in human capital.

--- Processing Page 16 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:

--- Processing Page 21 ---
Extracted Arguments:

--- Processing Page 23 ---
Extracted Arguments:

--- Processing Page 26 ---
Extracted Arguments:

--- Processing Page 30 ---
Extracted Arguments:

--- Processing Page 31 ---
Extracted Arguments:

--- Processing Page 32 ---


## SGD 6: Water and sanitation

In [ ]:
topic = "SGD 6 (Water and sanitation): Ensure availability and sustainable management of water and sanitation for all"
sgd_number = "6"
keywords_g6 = {
    "in_favor": [
        "clean water", "sanitation", "hygiene", "cleanliness", "sewerage",
        "drinking water", "water access", "water management", "water efficiency",
        "wastewater treatment", "water quality"
    ],
    "against": [
        "water scarcity", "water pollution", "lack of sanitation", "open defecation",
        "waterborne diseases", "drought", "unsustainable water use", "contaminated water"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g6)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 30 ---
Extracted Arguments:

--- Processing Page 31 ---
Extracted Arguments:
1. This is still above the 3 per cent SDG target, however.
2. To achieve the SDG target, stakeholders could leverage digitalization to reduce costs, increase efficiency and improve remittance accessibility.

--- Processing Page 32 ---
Extracted Arguments:

--- Processing Page 33 ---
Extracted Arguments:

--- Processing Page 34 ---
Extracted Arguments:

--- Processing Page 35 ---
Extracted Arguments:
1. Reporting is more common on environmental (water, emissions and energy efficiency) and governance (gender diversity, board meetings, bribery and corruption) dimensions than on social ones (human rights, health and safety, diversity and opportunity).

--- Processing Page 36 ---
Extracted Arguments:

--- 

## SGD 7: Clean Energy

In [ ]:
topic = "SGD 7 (Clean Energy): Ensure access to affordable, reliable, sustainable and modern energy for all"
sgd_number = "7"
keywords_g7 = {
    "in_favor": [
        "clean energy", "green energy", "renewable energy", "sustainable energy",
        "modern energy", "energy access", "energy efficiency", "solar power",
        "wind power", "geothermal energy", "hydropower", "energy transition", "energy matrix"
    ],
    "against": [
        "fossil fuels", "energy poverty", "energy inefficiency", "pollution",
        "carbon emissions", "unsustainable energy", "reliance on non-renewables"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g7)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 30 ---
Extracted Arguments:

--- Processing Page 31 ---
Extracted Arguments:
1. This is still above the 3 per cent SDG target, however.
2. To achieve the SDG target, stakeholders could leverage digitalization to reduce costs, increase efficiency and improve remittance accessibility.

--- Processing Page 32 ---
Extracted Arguments:

--- Processing Page 33 ---
Extracted Arguments:

--- Processing Page 34 ---
Extracted Arguments:

--- Processing Page 35 ---
Extracted Arguments:

--- Processing Page 36 ---
Extracted Arguments:
1. The energy sector, responsible for 86 per cent of global CO2 emissions, remains the largest contributor, driven by the expansion of coal- and gas-fired power generation.

--- Processing Page 37 ---
Extracted Arguments:
1. Fossil fuel subsidies reached a r

## SGD 8: Decent Work, Economic Growth

In [ ]:
topic = "SGD 8 (decent work, economic growth): Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all"
sgd_number = "8"
keywords_g8 = {
    "in_favor": [
        "decent work", "full employment", "fair wages", "workers rights",
        "job creation", "entrepreneurship", "financial inclusion", "financial",
        "business", "trade", "industrial", "commercial", "mercantile", "spillover"
    ],
    "against": [
        "unemployment", "underemployment", "precarious work", "exploitation",
        "child labor", "forced labor", "unsafe working conditions", "stagnation",
        "recession", "inequality", "informal economy", "low wages", "job insecurity",
        "informal jobs"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g8)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:
1. Employment and time use

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 30 ---
Extracted Arguments:
1. Earnings from work are particularly important for less well-off and vulnerable people.

--- Processing Page 31 ---
Extracted Arguments:
1. The global average cost of sending $200 decreased from 7.42 per cent in 2016 to 6.18 per cent in 2023. This is still above the 3 per cent SDG target, however.
2. To achieve the SDG target, stakeholders could leverage digitalization to reduce costs, increase efficiency and improve remittance accessibility.

--- Processing Page 32 ---
Extracted Arguments:

--- Processing Page 33 ---
Extracted Arguments:

--- Processing Page 34 ---
Extracted Arguments:

--- Processing Page 35 ---
Extracted Arguments:

--- Processing Page 36 ---
Extracted Arguments:

--- Processing Page 37 ---
Extracted 

## SGD 9: Infrastructure, industrilization, innovation

In [ ]:
topic = "SGD 9 (Infrastructure, industrilization, innovation): Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation"
sgd_number = "9"
keywords_g9 = {
    "in_favor": [
        "infrastructure", "industrialization", "innovation", "technological innovations",
        "research and development", "technology transfer", "connectivity", "internet access",
        "manufacturing", "scientific research", "digitalization", "modernization",
        "technological advances", "digital inclusion", "digital literacy", "technological investment" 
    ],
    "against": [
        "lack of infrastructure", "inadequate infrastructure", "industrial pollution",
        "unsustainable industry", "digital divide", "lack of innovation", "technological gap",
        "brain drain", "resource depletion", "unmaintained", "obsolescence", "decay", "cybersecurity threaths",
        "cybersecurity attacks"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g9)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:
1. Colombia and Malaysia have integrated geospatial and population data to estimate the proportion of the rural population living within 2 kilometres of an all-season road (SDG indicator 9.1.1).
2. Countries worldwide are recognizing the need to invest in national statistical systems to produce high-quality, timely data for SDG monitoring.
3. This involves not only financial resources but also requires building capacity, modernizing infrastructure and adopting international statistical standards.

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 30 ---
Extracted Arguments:

--- Processing Page 31 ---
Extracted Arguments:
1. Digital remittance services could help reach the target faster.
2. In 2023, the global average cost of digital remittances was 4.84 per cent compared to 6.77 per cent for non-digital/cash-based remittances

## SGD 10: Inequality

In [ ]:
topic = "SGD 10 (Inequality): Reduce inequality within and among countries"
sgd_number = "10"
keywords_g10 = {
    "in_favor": [
        "equality", "equity", "inclusion", "equal opportunity", "fairness",
        "social justice", "progressive taxation", "non-discrimination"
    ],
    "against": [
        "inequality", "disparity", "discrimination", "exclusion", "apartheid",
        "linguistic imperialism", "favouritism", "bias", "partiality", "injustice",
        "imbalance", "nepotism", "marginalization", "wealth concentration",
        "poverty gap", "social stratification", "prejudice"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g10)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:
1. Interestingly, high-income countries generally exhibited lower overall disaggregated data availability compared to low- and middle-income countries.

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:
1. Forty-five countries or areas received funding from donors; half were low- and lower-middle-income countries.

--- Processing Page 30 ---
Extracted Arguments:
1. Addressing inequality both within and among countries necessitates equitable resource distribution, investment in education and skills, social protection measures, efforts to stop discrimination, support for marginalized groups, and international cooperation for fair trade and financial systems.
2. Latin America and the Caribbean has high levels of within-country inequality with at least 18 per cent of the population living on less than half the median income.

--- Processing Page 31 --

## SGD 11: Sustainable cities

In [ ]:
topic = "SGD 11 (Sustainable Cities, Sustainable Communities): Make cities and human settlements inclusive, safe, resilient and sustainable"
sgd_number = "11"
keywords_g11 = {
    "in_favor": [
        "sustainable cities", "sustainable communities", "smart cities", "urban planning",
        "affordable housing", "public transport", "green spaces", "community",
        "preservation", "society", "people", "public", "association", "population",
        "residents", "commonwealth", "general public", "spatial justice", "accessibility"
    ],
    "against": [
        "slums", "urban sprawl", "air pollution", "noise pollution", "traffic",
        "lack of housing", "urban poverty", "crime", "segregation", "gentrification",
        "unsafe", "insecure", "urban degradation", "housing crisis", "urban decay",
        "deteriorated urban areas", "disadvantaged communities"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g11)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:
1. Both the Netherlands and Uruguay monitor air pollution with national networks of sensors (SDG indicator 11.6.2).

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 30 ---
Extracted Arguments:

--- Processing Page 31 ---
Extracted Arguments:

--- Processing Page 32 ---
Extracted Arguments:
1. Globally, approximately one quarter of the urban population lives in slums, with the total slum population reaching 1.1 billion in 2022.
2. The lack of equitable access to public transportation is a significant concern, particularly in LDCs, where only 4 in 10 individuals have convenient access.
3. Only 40 per cent of city dwellers can easily reach open public spaces.
4. With urbanization on the rise and nearly 70 per cent of the global population projected to reside in cities by 2050, the development of critical infrastructure, afforda

## SGD 12: Responsible Consumption, Responsible Production

In [ ]:
topic = "SGD 12 (Responsible Consumption, Responsible Production): Ensure sustainable consumption and production patterns"
sgd_number = "12"
keywords_g12 = {
    "in_favor": [
        "sustainable consumption", "sustainable production", "second use", "second hand",
        "circular economy", "recicle", "recycling", "reuse", "sustainable sourcing",
        "eco-design", "corporate social responsibility", "sustainable tourism",
        "manufacture", "manufacturing", "construction"
    ],
    "against": [
        "overconsumption", "waste", "using up", "expenditure", "exhaustion", "depletion",
        "dissipation", "pollution", "planned obsolescence", "fast fashion", "food waste",
        "unsustainable production", "resource inefficiency", "long-tail economy"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g12)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 30 ---
Extracted Arguments:

--- Processing Page 31 ---
Extracted Arguments:
1. To achieve the SDG target, stakeholders could leverage digitalization to reduce costs, increase efficiency and improve remittance accessibility.

--- Processing Page 32 ---
Extracted Arguments:

--- Processing Page 33 ---
Extracted Arguments:

--- Processing Page 34 ---
Extracted Arguments:
1. Achieving Goal 12 requires fostering circular economy models, sustainable production practices and responsible consumption.
2. These approaches can take advantage of opportunities at every stage of production to reduce resource and fossil fuel use, drive innovation, conserve energy and mitigate emissions.
3. High rates of consumption and insufficient reuse or recycling are producing vast piles of e-waste
4. I

## SGD 13: Climate change

In [ ]:
topic = "SGD 13 (Climate change): Take urgent action to combat climate change and its impacts"
sgd_number = "13"
keywords_g13 = {
    "in_favor": [
        "climate action", "mitigation", "adaptation", "resilience", "carbon neutrality",
        "decarbonization", "energy transition", "emissions reduction",
        "Paris Agreement", "climate policy"
    ],
    "against": [
        "climate change", "global warming", "greenhouse gas emissions", "CO2 emissions",
        "fossil fuels", "deforestation", "climate inaction", "climate denial",
        "extreme weather events", "sea-level rise", "environmental degradation"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g13)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:
1. Moreover, data timeliness remains a challenge.
2. with major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16).

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:
1. countries agreed on “an urgent and sustained increase in the level and scale of investments in data and statistics from domestic and international actors, from the public, private and philanthropic sectors, to strengthen statistical capacity in low-income countries and fragile states, close data gaps for vulnerable groups and enhance country resilience in the current context of economic crisis, conflict, climate change and increased food insecurity.”

--- Processing Page 30 ---
Extracted Arguments:

--- Processing Page 31 ---
Extracted Arguments:

--- Processing Page 32 ---
Extr

## SGD 14: Life bellow water

In [ ]:
topic = "SGD 14 (Life bellow Water): Conserve and sustainably use the oceans, seas and marine resources for sustainable development"
sgd_number = "14"
keywords_g14 = {
    "in_favor": [
        "ocean conservation", "marine conservation", "sustainable fishing",
        "marine protected areas", "ocean biodiversity", "ocean ecosystems", "biology",
        "marine biology", "ecosystem restoration"
    ],
    "against": [
        "overfishing", "marine pollution", "plastic pollution", "microplastics",
        "ocean acidification", "coral bleaching", "habitat destruction", "illegal fishing",
        "destructive fishing practices", "biodiversity loss", "eutrophication"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g14)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:
1. Ghana and the United Kingdom have tapped into citizen science data to monitor marine litter (SDG indicator 14.1.1).

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 30 ---
Extracted Arguments:

--- Processing Page 31 ---
Extracted Arguments:

--- Processing Page 32 ---
Extracted Arguments:

--- Processing Page 33 ---
Extracted Arguments:

--- Processing Page 34 ---
Extracted Arguments:

--- Processing Page 35 ---
Extracted Arguments:

--- Processing Page 36 ---
Extracted Arguments:

--- Processing Page 37 ---
Extracted Arguments:

--- Processing Page 38 ---
Extracted Arguments:
1. Oceans face significant challenges from eutrophication, worsening acidification, declining fish stocks, rising temperatures and widespread pollution.
2. All these factors destroy habitats, diminish biodiversity and threaten coastal communities a

## SGD 15: Life on land

In [ ]:
topic = "SGD 15 (Life on land): Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss"
sgd_number = "15"
keywords_g15 = {
    "in_favor": [
        "land ecosystem", "agriculture", "ecosystem restoration", "forest",
        "stop desertification", "reverse land degradation", "conservation",
        "sustainable agriculture", "afforestation", "reforestation",
        "wildlife protection", "wildlife"
    ],
    "against": [
        "deforestation", "desertification", "land degradation", "biodiversity loss",
        "habitat loss", "poaching", "illegal wildlife trade", "invasive species",
        "soil erosion", "unsustainable agriculture", "soil pollution"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g15)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:
1. For instance, Azerbaijan uses remote sensing to monitor the coverage of important sites for mountain biodiversity (SDG indicator 15.4.1).

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 30 ---
Extracted Arguments:

--- Processing Page 31 ---
Extracted Arguments:

--- Processing Page 32 ---
Extracted Arguments:
1. This unregulated growth has negative impacts on the natural environment and land use, contributing to increased air pollution and the loss of open spaces, wildlife habitats and agricultural land.

--- Processing Page 33 ---
Extracted Arguments:

--- Processing Page 34 ---
Extracted Arguments:

--- Processing Page 35 ---
Extracted Arguments:

--- Processing Page 36 ---
Extracted Arguments:

--- Processing Page 37 ---
Extracted Arguments:

--- Processing Page 38 ---
Extracted Arguments:

--- Processing Page 39 ---

## SGD 16: Peace, Justice, Strong Institutions

In [ ]:
topic = "SGD 16 (Peace, Justice, Strong Institutions): Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels"
sgd_number = "16"
keywords_g16 = {
    "in_favor": [
        "peace", "justice", "access to justice", "strong institutions", "healthy institutions",
        "accountability", "anti-corruption", "transparency", "governance", "human rights",
        "conflict resolution", "truce", "ceasefire", "treaty", "armistice", "pacification",
        "fairness","integrity",
        "honesty", "decency", "impartiality", "justness", "rightfulness",
        "strong leadership", "good leadership", "institutionalization", "government effort", 
        "public investments", "science-based policy"

    ],

    "against": [
        "conflict", "violence", "war", "insecurity", "injustice", "corruption", "bribery",
        "weak institutions", "lack of accountability", "impunity", "human rights violations",
        "discrimination", "crime", "illicit financial flows", "organized crime", "terrorism",
         "weak leadership", "autoritarism", "dictator"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g16)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:
1. Moreover, data timeliness remains a challenge.
2. with major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16).

--- Processing Page 7 ---
Extracted Arguments:
1. Crime and justice

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 30 ---
Extracted Arguments:

--- Processing Page 31 ---
Extracted Arguments:
1. By end 2023, a record 37.4 million refugees under the mandate of the United Nations High Commissioner for Refugees (UNHCR) remained forcibly displaced from their countries due to war, conflict, persecution, human rights violations and events seriously disturbing public order.
2. In 2023, a tragic milestone occurred as it became the deadliest year on record for migrants, with 8,177 fatalities documented.

--- Processing Page 32 ---
Extracted Arguments:

--- Proce

## SGD 17: Partnerships, sustainable development

In [ ]:
topic = "SGD 17 (Partnerships, sustainable development): Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development"
sgd_number = "17"
keywords_g17 = {
    "in_favor": [
        "global partnership", "cooperation", "association", "alliance", "sharing",
        "union", "connection", "participation", "copartnership", "technology transfer",
        "capacity building", "international cooperation", 'positive spillover', 
        "transboundary", "coordination"
    ],
    "against": [
        "lack of cooperation", "isolationism", "protectionism", "insufficient funding",
        "debt", "policy incoherence", "data gaps", "weak monitoring", "non-participation",
        "aid dependency", "technological gatekeeping", "negative spillover"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g17)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:
1. As countries continue to strengthen statistical foundations, embracing innovation and integrating diverse data sources and methodologies will be critical in overcoming challenges such as declining response rates as well as in fostering collaborative partnerships.

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:
1. Strengthening partnerships is key to more inclusive data.
2. Building partnerships with diverse stakeholders makes SDG monitoring more inclusive and incorporates different perspectives and needs.
3. Partnerships between NSOs and civil society organizations stood at 37 per cent.
4. As more countries recognize the importance of adopting a “whole-of-society” approach to achieve the ambitious goals of the 2030 Agenda, increased efforts are being made to acknowledge the contributions of civil society.
5. This initiative represents a sign

## SGD 0: Overarching terms

In [ ]:
topic = "SGD Overarching terms: Sustainable Development Goal, SDG, Agenda 2030, leave no one behind, Voluntary National Review, SDG transformations, "
sgd_number = "0"
keywords_g0 = {
    "in_favor": [
        "Sustainability", "Sustainable Development Goal", "SDG", "Agenda 2030", "global goals", 
        "development", "progress", "implementation", "monitoring", "accountability", "inclusive", "leave no one behind", 
        "Voluntary National Review", "VNR", "SDG transformations"
    ],
    "against": [
          "Unsustainability", "inaction", "regression", "lack of funding", "greenwashing", "exploitation", 
          "environmental degradation", "SDG needs", "regression", 
          "multidimensional vulnerability", "stagnation"
    ]
}

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  keywords = keywords_g0)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Extracted Arguments:
1. Overall progress across targets based on 2015–2024 global aggregate data
2. Despite commendable increases in data to monitor the SDGs, critical gaps persist

--- Processing Page 7 ---
Extracted Arguments:
1. Engaging citizens in data production is essential to leave no one behind.
2. The overarching principle of the 2030 Agenda for Sustainable Development is to leave no one behind.

--- Processing Page 8 ---
Extracted Arguments:
1. Ensuring that no one is left behind calls for more than just data disaggregation.
2. Since the adoption of the 2030 Agenda, countries have made significant progress in opening up official statistics for public use.
3. Successful SDG monitoring requires NSOs to play a strong stewardship role within the national data ecosystem.

--- Processing Page 9 ---
Extracted Arguments:
1. As more countries recognize the importance of adopting a “whole-of-society” approach to achieve the ambitious goals of the 2030 Agenda